# Mini-CLIP on a real dataset (Flickr8k): image encoder = ResNet-18, text encoder = Transformer, loss = −log softmax (InfoNCE)

Every section ends with plots (saved to `images/`). Order:
1. setup → 2. data → 3. tokenizer + loaders → 4. model → 5. **before training** (embedding space, feature maps, attention) → 6. training → 7. curves → 8. test retrieval → 9. similarity structure → 10. retrieval galleries → 11. embedding space **2D** → 12. embedding space **3D** (static + interactive + animation) → 13. embedding diagnostics → 14. interpretability (attention, Grad-CAM, prompts, zero-shot) → 15. summary + save.

Run on a Colab GPU (Runtime → Change runtime type → T4). Set `PRETRAINED = False` below to train the CNN from scratch.

In [ ]:
%pip install -q datasets umap-learn plotly

In [ ]:
import os, random, time, copy, json, math, re, textwrap, zipfile, shutil
from collections import OrderedDict, Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet18 as tv_resnet18, ResNet18_Weights

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from scipy.cluster import hierarchy
from scipy.linalg import orthogonal_procrustes

import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3D projection)

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False
    print("umap-learn not available -> UMAP plots will be skipped")

# ------------------------------- config -------------------------------
SEED = 42
DATASET = "jxie/flickr8k"        # 8k photos x 5 captions (HuggingFace)
PRETRAINED = True                # True: ImageNet-pretrained ResNet-18 | False: train the CNN from scratch
IMG_SIZE = 224                   # use 128 if PRETRAINED = False (faster)
CACHE_SIZE = 256                 # every image is resized ONCE to this and kept in RAM (use 144 if IMG_SIZE = 128)
EMBED_DIM = 256                  # size of the shared image/text space
TXT_LAYERS = 2                   # transformer blocks in the text encoder
TXT_HEADS = 4                    # attention heads (EMBED_DIM must be divisible by this)
MAX_LEN = 32                     # max tokens per caption, including [CLS]
MIN_FREQ = 3                     # words rarer than this become [UNK]
BATCH_SIZE = 128                 # bigger batch = more negatives per positive
NUM_WORKERS = 2
LR = 3e-4
WEIGHT_DECAY = 0.05
INIT_TEMP = 0.07                 # starting temperature (learned afterwards)
N_VIS = 600                      # how many val/test pairs are drawn in embedding plots

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 20 if DEVICE.type == "cuda" else 2
USE_AMP = DEVICE.type == "cuda"
SNAPSHOT_EPOCHS = sorted({e for e in (1, 2, 4, 8, 12, EPOCHS) if e <= EPOCHS})   # embedding space is saved at these epochs

if DEVICE.type != "cuda":
    print("WARNING: no GPU. EPOCHS=2 just to test the pipeline.")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

COLORS = {"image": "tab:blue", "text": "tab:orange"}
PL_COLORS = {"image": "#1f77b4", "text": "#ff7f0e"}      # same colors for plotly

os.makedirs("images", exist_ok=True)
def save_fig(name):
    plt.savefig(f"images/{name}.png", dpi=130, bbox_inches="tight")

print("Device:", DEVICE, "| image size:", IMG_SIZE, "| epochs:", EPOCHS, "| AMP:", USE_AMP, "| pretrained:", PRETRAINED)

## 2. Data: Flickr8k photos + 5 human captions each
We split **by image** (never by caption) so the same photo can't be in both train and test. Photos are resized once and cached in RAM, so the GPU isn't waiting on JPEG decoding.

Flickr8k has no class labels, so for *coloring plots only* we derive a rough "theme" per image from keywords in its 5 captions (dog, water, snow, ...). The model never sees this label.

In [ ]:
from datasets import load_dataset

def tokenize(s):
    return re.findall(r"[a-z0-9']+", s.lower())

raw = load_dataset(DATASET)
print(raw)

def pick_splits(raw):
    keys = list(raw.keys())
    val_key = next((k for k in keys if k.startswith("val")), None)
    if "train" in keys and "test" in keys and val_key:
        return raw["train"], raw[val_key], raw["test"]
    base = raw[keys[0]]                                      # single-split datasets: make our own 80/10/10
    perm = np.random.RandomState(SEED).permutation(len(base)).tolist()
    n = len(base); a, b = int(0.8 * n), int(0.9 * n)
    return base.select(perm[:a]), base.select(perm[a:b]), base.select(perm[b:])

hf = dict(zip(["train", "val", "test"], pick_splits(raw)))
IMG_COL = next(c for c in ("image", "img", "jpg", "photo") if c in hf["train"].column_names)

def extract_captions(ds):
    cols = [c for c in ds.column_names if c.lower().startswith(("caption", "sentence", "text"))]
    per_col = [ds[c] for c in cols]
    out = []
    for i in range(len(ds)):
        caps_i = []
        for col in per_col:
            v = col[i]
            caps_i += [v] if isinstance(v, str) else list(v)
        out.append([c.strip() for c in caps_i if c and c.strip()])
    return out

caps = {s: extract_captions(hf[s]) for s in hf}

# Resize + center-crop once, keep PIL images in RAM
resize_once = T.Compose([T.Resize(CACHE_SIZE), T.CenterCrop(CACHE_SIZE)])
def preload(ds):
    return [resize_once(ds[i][IMG_COL].convert("RGB")) for i in tqdm(range(len(ds)), leave=False)]

t0 = time.time()
imgs_pil = {s: preload(hf[s]) for s in hf}
print(f"Loaded + resized in {time.time()-t0:.0f}s |", {s: f"{len(imgs_pil[s])} images x {len(caps[s][0])} captions" for s in hf})

# ---------- keyword themes (ONLY for coloring plots / sanity checks) ----------
CATEGORIES = OrderedDict([
    ("dog",        {"dog", "dogs", "puppy", "puppies"}),
    ("water",      {"water", "beach", "ocean", "pool", "swimming", "swims", "wave", "waves", "lake", "river", "surf", "sea", "splashing"}),
    ("snow",       {"snow", "snowy", "ski", "skis", "skier", "skiing", "snowboard", "snowboarder", "sled"}),
    ("ball sport", {"ball", "soccer", "football", "basketball", "baseball", "tennis", "volleyball", "goalie", "player", "players"}),
    ("vehicle",    {"bike", "bicycle", "motorcycle", "motorbike", "car", "truck", "biker", "cyclist", "bmx"}),
    ("climbing",   {"mountain", "rock", "cliff", "climbing", "climbs", "hiking", "hill", "forest", "trail"}),
    ("child",      {"child", "children", "boy", "boys", "girl", "girls", "kid", "kids", "baby", "toddler"}),
    ("crowd",      {"crowd", "people", "group", "street", "city"}),
    ("adult",      {"man", "woman", "men", "women", "lady", "person"}),
])
CAT_NAMES = list(CATEGORIES) + ["other"]
CAT_COLOR = {c: plt.cm.tab10(i) for i, c in enumerate(CAT_NAMES)}

def image_category(cap_list):
    counts = Counter()
    for c in cap_list:
        words = set(tokenize(c))
        for name, kw in CATEGORIES.items():
            if words & kw:
                counts[name] += 1
    if not counts:
        return "other"
    top = max(counts.values())
    return next(n for n in CATEGORIES if counts.get(n, 0) == top)

cats = {s: np.array([image_category(c) for c in caps[s]]) for s in hf}

# ---------- visuals ----------
rng = np.random.RandomState(0)
pick = rng.choice(len(imgs_pil["train"]), 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax, i in zip(axes.ravel(), pick):
    ax.imshow(imgs_pil["train"][i]); ax.axis("off")
    ax.set_title(f"[{cats['train'][i]}]\n" + textwrap.fill(caps["train"][i][0], 38), fontsize=8)
plt.suptitle("Flickr8k: 12 random training photos (title = theme + first caption)", fontsize=13)
plt.tight_layout(); save_fig("01_samples"); plt.show()

# 5 captions of one photo
i0 = int(pick[0])
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.imshow(imgs_pil["train"][i0]); ax.axis("off")
ax.set_title("One photo, 5 different human descriptions (the model sees ONE random one per epoch)", fontsize=10)
ax.text(1.02, 0.5, "\n\n".join(textwrap.fill(c, 45) for c in caps["train"][i0][:5]), transform=ax.transAxes, va="center", fontsize=9)
plt.tight_layout(); save_fig("01b_five_captions"); plt.show()

# caption statistics
all_tr = [c for cl in caps["train"] for c in cl]
lens = [len(tokenize(c)) for c in all_tr]
wc = Counter(w for c in all_tr for w in tokenize(c))
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].hist(lens, bins=range(0, max(lens) + 2), color="tab:orange"); ax[0].axvline(MAX_LEN - 1, color="red", ls="--", label=f"MAX_LEN-1 = {MAX_LEN-1}")
ax[0].set_title("Caption length (words)"); ax[0].set_xlabel("words"); ax[0].legend()
top = wc.most_common(20)
ax[1].barh([w for w, _ in top][::-1], [c for _, c in top][::-1], color="tab:orange"); ax[1].set_title("20 most frequent words")
freqs = np.array(sorted(wc.values(), reverse=True))
ax[2].loglog(np.arange(1, len(freqs) + 1), freqs, color="tab:purple"); ax[2].axhline(MIN_FREQ, color="red", ls="--", label=f"MIN_FREQ = {MIN_FREQ}")
ax[2].set_title("Word frequency (Zipf curve)"); ax[2].set_xlabel("rank"); ax[2].legend()
plt.tight_layout(); save_fig("02_caption_stats"); plt.show()

# theme balance
x, w = np.arange(len(CAT_NAMES)), 0.27
fig, ax = plt.subplots(figsize=(13, 4))
for k, s in enumerate(["train", "val", "test"]):
    cnt = np.array([(cats[s] == c).mean() for c in CAT_NAMES])
    ax.bar(x + (k - 1) * w, cnt, w, label=s)
ax.set_xticks(x); ax.set_xticklabels(CAT_NAMES, rotation=30, ha="right"); ax.set_ylabel("fraction of images")
ax.set_title("Keyword themes (used only to color plots)"); ax.legend()
plt.tight_layout(); save_fig("03_theme_distribution"); plt.show()

## 3. Tokenizer, augmentation, loaders
- Word-level vocabulary from the **train** captions only. A `[CLS]` token goes in front of each caption; its final vector becomes the caption embedding.
- Each epoch an image is paired with **one random** of its 5 captions. So an image appears once per batch, which means no accidental "false negatives".
- No horizontal flip: it would turn "left" into "right" and make the caption wrong.

In [ ]:
PAD, CLS, UNK = 0, 1, 2
itos = ["[PAD]", "[CLS]", "[UNK]"] + [w for w, c in wc.most_common() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_SIZE = len(itos)

def encode(s):
    ids = [CLS] + [stoi.get(w, UNK) for w in tokenize(s)][:MAX_LEN - 1]
    return torch.tensor(ids, dtype=torch.long)

def pad_batch(list_of_ids):
    return nn.utils.rnn.pad_sequence(list_of_ids, batch_first=True, padding_value=PAD)

unk_rate = sum(stoi.get(w, UNK) == UNK for c in all_tr for w in tokenize(c)) / sum(lens)
print(f"vocab size {VOCAB_SIZE} | words mapped to [UNK]: {unk_rate:.2%}")
for c in caps["train"][i0][:2]:
    ids = encode(c)
    print(f"\n'{c}'")
    print("  ids   :", ids.tolist())
    print("  tokens:", [itos[t] for t in ids.tolist()])

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.9, 1.1)),
    T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
test_tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE), T.ToTensor(), T.Normalize(MEAN, STD)])

def denorm(t):
    m = torch.tensor(MEAN).view(3, 1, 1); s = torch.tensor(STD).view(3, 1, 1)
    return (t.cpu().float() * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

class PairDataset(Dataset):
    """One (image, random caption) pair per image."""
    def __init__(self, imgs, caps, tf, train=True):
        self.imgs, self.caps, self.tf, self.train = imgs, caps, tf, train
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, i):
        cap = random.choice(self.caps[i]) if self.train else self.caps[i][0]
        return self.tf(self.imgs[i]), encode(cap), i

def collate(batch):
    imgs, ids, idx = zip(*batch)
    return torch.stack(imgs), pad_batch(list(ids)), torch.tensor(idx)

loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0, collate_fn=collate)
train_ds = PairDataset(imgs_pil["train"], caps["train"], train_tf, train=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kw)
print("train images:", len(train_ds), "| batches/epoch:", len(train_loader), "| chance-level loss = ln(batch) =", round(math.log(BATCH_SIZE), 3))

# One photo, 9 training views (augmentation)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0, 0].imshow(denorm(test_tf(imgs_pil["train"][i0]))); axes[0, 0].set_title("ORIGINAL (test view)", color="red")
for ax in axes.ravel()[1:]:
    ax.imshow(denorm(train_tf(imgs_pil["train"][i0]))); ax.set_title("augmented")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Training views of one photo (random crop + color jitter)")
plt.tight_layout(); save_fig("04_augmentation"); plt.show()

# What a batch looks like after padding
imgs_b, ids_b, idx_b = next(iter(train_loader))
print("image batch:", tuple(imgs_b.shape), "| token batch:", tuple(ids_b.shape), "(padded with 0 to the longest caption in the batch)")
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow((ids_b[:16] == PAD).numpy(), cmap="Greys", aspect="auto"); ax.set_yticks([]); ax.set_xlabel("token position")
ax.set_title("Padding mask of 16 captions (black = padding, ignored by attention)")
plt.tight_layout(); save_fig("05_padding_mask"); plt.show()

## 4. The model
**Image encoder** = ResNet-18 written by hand (same as your ResNet notebook, with torchvision's layer names so pretrained weights load directly) → global average pool (512) → Linear → L2-normalize.
**Text encoder** = token + position embeddings → `TXT_LAYERS` pre-LN Transformer blocks (multi-head self-attention + MLP, padding masked) → take the `[CLS]` vector → LayerNorm → Linear → L2-normalize.
**Loss** = softmax over the batch similarity matrix, then −log of the correct entry (cross-entropy), in both directions. The temperature is a learned parameter.

In [ ]:
class BasicBlock(nn.Module):
    """conv3x3-BN-ReLU-conv3x3-BN, ADD the input (skip connection), ReLU."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class ResNet18Backbone(nn.Module):
    """ResNet-18 without the classification layer. Layer names match torchvision."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1 = nn.Sequential(BasicBlock(64, 64), BasicBlock(64, 64))
        self.layer2 = nn.Sequential(BasicBlock(64, 128, 2), BasicBlock(128, 128))
        self.layer3 = nn.Sequential(BasicBlock(128, 256, 2), BasicBlock(256, 256))
        self.layer4 = nn.Sequential(BasicBlock(256, 512, 2), BasicBlock(512, 512))
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def forward(self, x, return_trace=False):
        trace = {"input": x}
        x = self.relu(self.bn1(self.conv1(x))); trace["stem"] = x
        x = self.maxpool(x)
        for name in ("layer1", "layer2", "layer3", "layer4"):
            x = getattr(self, name)(x); trace[name] = x
        x = torch.flatten(self.avgpool(x), 1); trace["gap"] = x
        return (x, trace) if return_trace else x

class ImageEncoder(nn.Module):
    def __init__(self, dim=EMBED_DIM, pretrained=PRETRAINED):
        super().__init__()
        self.backbone = ResNet18Backbone()
        if pretrained:
            sd = tv_resnet18(weights=ResNet18_Weights.DEFAULT).state_dict()
            missing, unexpected = self.backbone.load_state_dict(sd, strict=False)
            assert not missing and all(k.startswith("fc.") for k in unexpected), (missing, unexpected)
            print("Loaded ImageNet weights into the hand-built ResNet-18 (only fc.* skipped).")
        self.proj = nn.Linear(512, dim)
    def forward(self, x, return_trace=False):
        feat, trace = self.backbone(x, return_trace=True)
        z = F.normalize(self.proj(feat), dim=-1)
        trace["embedding"] = z
        return (z, trace) if return_trace else z

class TextBlock(nn.Module):
    """Pre-LN Transformer block: x + MHA(LN(x)), then x + MLP(LN(x))."""
    def __init__(self, dim, heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim), nn.Dropout(dropout))
        self.last_attn = None
    def forward(self, x, pad_mask, keep_attn=False):
        h = self.ln1(x)
        a, w = self.attn(h, h, h, key_padding_mask=pad_mask, need_weights=keep_attn, average_attn_weights=False)
        if keep_attn:
            self.last_attn = w.detach()                  # N x heads x L x L
        x = x + a
        return x + self.mlp(self.ln2(x))

class TextEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, dim=EMBED_DIM, heads=TXT_HEADS, layers=TXT_LAYERS, max_len=MAX_LEN):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, dim, padding_idx=PAD)
        self.pos = nn.Embedding(max_len, dim)
        self.blocks = nn.ModuleList([TextBlock(dim, heads) for _ in range(layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.proj = nn.Linear(dim, dim)
        nn.init.normal_(self.tok.weight, std=0.02); nn.init.normal_(self.pos.weight, std=0.01)
    def forward(self, ids, return_trace=False, keep_attn=False):
        N, L = ids.shape
        pad_mask = ids == PAD                                          # True = ignore
        x = self.tok(ids) + self.pos(torch.arange(L, device=ids.device))
        trace = {"embed": x}
        for i, blk in enumerate(self.blocks):
            x = blk(x, pad_mask, keep_attn); trace[f"block{i+1}"] = x
        cls_vec = self.ln_f(x[:, 0]); trace["cls"] = cls_vec
        z = F.normalize(self.proj(cls_vec), dim=-1)
        trace["embedding"] = z
        return (z, trace) if return_trace else z

class CLIP(nn.Module):
    def __init__(self):
        super().__init__()
        self.image = ImageEncoder()
        self.text = TextEncoder()
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / INIT_TEMP)))   # scale = 1 / temperature, learned
    def forward(self, imgs, ids):
        return self.image(imgs), self.text(ids), self.logit_scale.exp()

def clip_loss(img_emb, txt_emb, scale):
    """Symmetric InfoNCE = average of two cross-entropies (softmax followed by -log of the diagonal)."""
    logits = scale * img_emb.float() @ txt_emb.float().T            # N x N, entry (i,j) = scaled cosine(image i, caption j)
    target = torch.arange(len(img_emb), device=img_emb.device)
    return (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2, logits

def count_params(m):
    return sum(p.numel() for p in m.parameters())

torch.manual_seed(SEED)
model = CLIP().to(DEVICE)

# Sanity checks
tv_no_fc = count_params(tv_resnet18()) - (512 * 1000 + 1000)
assert count_params(model.image.backbone) == tv_no_fc, "hand-built ResNet-18 differs from torchvision"
with torch.no_grad():
    ie_, te_ = torch.randn(6, 8), torch.randn(6, 8)
    ie_, te_ = F.normalize(ie_, dim=-1), F.normalize(te_, dim=-1)
    loss_ce, lg = clip_loss(ie_, te_, torch.tensor(14.3))
    manual = -(torch.log_softmax(lg, 1).diag().mean() + torch.log_softmax(lg.T, 1).diag().mean()) / 2
print(f"cross_entropy loss {loss_ce.item():.6f} == manual -log(softmax) {manual.item():.6f}")

arch = {"image encoder": count_params(model.image), "text encoder": count_params(model.text), "total": count_params(model)}
print({k: f"{v:,}" for k, v in arch.items()})
print("initial temperature:", INIT_TEMP, "-> scale", round(model.logit_scale.exp().item(), 2))

# Shapes through both encoders
with torch.no_grad():
    model.eval()
    _, tr_i = model.image(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE), return_trace=True)
    _, tr_t = model.text(ids_b[:2].to(DEVICE), return_trace=True)
print("\nImage path:"); [print(f"  {k:9s} -> {tuple(v.shape)}") for k, v in tr_i.items()]
print("Text path:");  [print(f"  {k:9s} -> {tuple(v.shape)}") for k, v in tr_t.items()]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
bars = ax[0].bar(list(arch), [v / 1e6 for v in arch.values()], color=["tab:blue", "tab:orange", "gray"]); ax[0].bar_label(bars, fmt="%.2f"); ax[0].set_title("Parameters (millions)")
n_layer = [("stem", count_params(nn.Sequential(model.image.backbone.conv1, model.image.backbone.bn1)))] + [(f"layer{i}", count_params(getattr(model.image.backbone, f"layer{i}"))) for i in range(1, 5)]
ax[1].bar([n for n, _ in n_layer], [c / 1e6 for _, c in n_layer], color="tab:blue"); ax[1].set_title("Where the ResNet-18 parameters live (M)")
plt.suptitle("Mini-CLIP size"); plt.tight_layout(); save_fig("06_architecture"); plt.show()

## 5. Before training: what does the (untrained) embedding space look like?
Helpers first: embed the whole val/test set, compute retrieval metrics (Recall@K, median rank), and reusable plot functions. Then we look at the **random** space: matched image/caption pairs should be unrelated, the similarity matrix should be noise, and the positive/negative similarity histograms should overlap completely.

(With `PRETRAINED=True` the CNN features are already meaningful, but the projection into the shared space and the whole text encoder are random, so image and text vectors still live in unrelated places.)

In [ ]:
# ---------------- evaluation data: all images + ALL 5 captions per image ----------------
def prep_eval(split):
    texts, t2i, first = [], [], []
    for i, cl in enumerate(caps[split]):
        first.append(len(texts))
        for c in cl:
            texts.append(c); t2i.append(i)
    return dict(imgs=imgs_pil[split], texts=texts, t2i=torch.tensor(t2i), first=torch.tensor(first), cats=cats[split])
EV = {s: prep_eval(s) for s in ("val", "test")}
VIS_IDX = {s: np.random.RandomState(SEED).choice(len(EV[s]["imgs"]), min(N_VIS, len(EV[s]["imgs"])), replace=False) for s in EV}

@torch.no_grad()
def embed_images(model, imgs, bs=256):
    model.eval(); out = []
    for i in range(0, len(imgs), bs):
        x = torch.stack([test_tf(im) for im in imgs[i:i + bs]]).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out.append(model.image(x).float().cpu())
    return torch.cat(out)

@torch.no_grad()
def embed_texts(model, texts, bs=512):
    model.eval(); out = []
    for i in range(0, len(texts), bs):
        ids = pad_batch([encode(t) for t in texts[i:i + bs]]).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out.append(model.text(ids).float().cpu())
    return torch.cat(out)

def retrieval_metrics(img_emb, txt_emb, t2i):
    sim = img_emb @ txt_emb.T                                         # (Ni, Nt)
    Ni = len(img_emb)
    order = sim.argsort(dim=1, descending=True)                       # image -> captions
    hit = t2i[order] == torch.arange(Ni)[:, None]
    rank_i2t = hit.float().argmax(1)                                  # 0-based rank of the first correct caption
    order_t = sim.T.argsort(dim=1, descending=True)                   # caption -> images
    rank_t2i = (order_t == t2i[:, None]).float().argmax(1)
    res = {}
    for k in (1, 5, 10):
        res[f"i2t_R@{k}"] = (rank_i2t < k).float().mean().item()
        res[f"t2i_R@{k}"] = (rank_t2i < k).float().mean().item()
    res["i2t_medr"] = rank_i2t.float().median().item() + 1
    res["t2i_medr"] = rank_t2i.float().median().item() + 1
    res["rsum"] = sum(res[f"{d}_R@{k}"] for d in ("i2t", "t2i") for k in (1, 5, 10))
    return res, rank_i2t, rank_t2i

def evaluate_model(model, split):
    ie = embed_images(model, EV[split]["imgs"]); te = embed_texts(model, EV[split]["texts"])
    res, r_i, r_t = retrieval_metrics(ie, te, EV[split]["t2i"])
    return res, ie, te, r_i, r_t

def make_snapshot(split, ie, te):
    """A small paired subset (image i + its first caption) used for all embedding-space plots."""
    idx = VIS_IDX[split]; first = EV[split]["first"][idx]
    return dict(img=ie[idx], txt=te[first], cat=EV[split]["cats"][idx], caps=[EV[split]["texts"][j] for j in first.tolist()], idx=idx)

def pos_neg(img, txt):
    sim = img @ txt.T
    pos = sim.diag()
    neg = sim[~torch.eye(len(sim), dtype=torch.bool)]
    return pos.numpy(), neg.numpy()

@torch.no_grad()
def val_loss_from_emb(ie, te, scale, split="val"):
    te1 = te[EV[split]["first"]]
    perm = torch.randperm(len(ie), generator=torch.Generator().manual_seed(0))
    losses = [clip_loss(ie[perm[i:i + BATCH_SIZE]], te1[perm[i:i + BATCH_SIZE]], scale)[0].item()
              for i in range(0, len(perm) - BATCH_SIZE + 1, BATCH_SIZE)]
    return float(np.mean(losses))

# ---------------- reusable plot functions ----------------
def plot_posneg(ax, img, txt, title):
    pos, neg = pos_neg(img, txt)
    ax.hist(neg, bins=60, alpha=0.6, color="tab:gray", density=True, label=f"mismatched pairs (mean {neg.mean():.2f})")
    ax.hist(pos, bins=40, alpha=0.7, color="tab:green", density=True, label=f"matched pairs (mean {pos.mean():.2f})")
    ax.set_xlabel("cosine similarity"); ax.set_title(title, fontsize=10); ax.legend(fontsize=8)

def plot_simmat(ax, img, txt, n=40, title=""):
    sim = (img[:n] @ txt[:n].T).numpy()
    im = ax.imshow(sim, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xlabel("caption j"); ax.set_ylabel("image i"); ax.set_title(title, fontsize=10)
    return im

def cat_legend(ax, cats_present=None, loc="best"):
    names = [c for c in CAT_NAMES if cats_present is None or c in set(cats_present)]
    h = [Line2D([], [], marker="s", ls="", color=CAT_COLOR[c], label=c) for c in names]
    h += [Line2D([], [], marker="o", ls="", color="k", label="image"), Line2D([], [], marker="^", ls="", color="k", label="text")]
    ax.legend(handles=h, fontsize=7, loc=loc, ncol=2)

def scatter_joint(ax, ci, ct, cats_, mode="modality", n_lines=40, title=""):
    if n_lines:
        for i in np.random.RandomState(0).choice(len(ci), min(n_lines, len(ci)), replace=False):
            ax.plot([ci[i, 0], ct[i, 0]], [ci[i, 1], ct[i, 1]], color="gray", lw=0.5, alpha=0.6, zorder=1)
    if mode == "modality":
        ax.scatter(ci[:, 0], ci[:, 1], s=14, c=COLORS["image"], marker="o", alpha=0.75, zorder=2, label="image")
        ax.scatter(ct[:, 0], ct[:, 1], s=24, c=COLORS["text"], marker="^", alpha=0.75, zorder=2, label="text")
        ax.legend(fontsize=7, loc="best")
    else:
        col = np.array([CAT_COLOR[c] for c in cats_])
        ax.scatter(ci[:, 0], ci[:, 1], s=14, c=col, marker="o", alpha=0.8, zorder=2)
        ax.scatter(ct[:, 0], ct[:, 1], s=26, c=col, marker="^", alpha=0.8, zorder=2)
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

_RED = {}
def reduce_joint(img, txt, method, n_components=2, key=None, seed=SEED):
    """Project image+text embeddings TOGETHER into 2D/3D so they share axes. Cached by key."""
    if key is not None and (key, method, n_components) in _RED:
        return _RED[(key, method, n_components)]
    X = np.concatenate([img.numpy(), txt.numpy()], 0)
    if method == "pca":
        Z = PCA(n_components, random_state=seed).fit_transform(X)
    elif method == "tsne":
        Z = TSNE(n_components, perplexity=min(30, (len(X) - 1) / 3), init="pca", learning_rate="auto", random_state=seed).fit_transform(X)
    elif method == "umap":
        Z = umap.UMAP(n_components=n_components, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=seed).fit_transform(X)
    out = (Z[:len(img)], Z[len(img):])
    if key is not None:
        _RED[(key, method, n_components)] = out
    return out

LAYERS = ["stem", "layer1", "layer2", "layer3", "layer4"]

def show_feature_maps(model, img_tensor, title, n_maps=8):
    model.eval()
    with torch.no_grad():
        _, trace = model.image(img_tensor.unsqueeze(0).to(DEVICE), return_trace=True)
    fig, axes = plt.subplots(len(LAYERS) + 1, n_maps, figsize=(14, 1.9 * (len(LAYERS) + 1)))
    axes[0, 0].imshow(denorm(img_tensor)); axes[0, 0].set_title("INPUT", fontsize=9)
    for ax in axes[0]:
        ax.axis("off")
    for r, lname in enumerate(LAYERS, 1):
        fmap = trace[lname][0].float().cpu()
        order = fmap.abs().mean(dim=(1, 2)).argsort(descending=True)[:n_maps]
        for c in range(n_maps):
            ax = axes[r, c]; ax.imshow(fmap[order[c]], cmap="viridis"); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"ch {order[c].item()}", fontsize=7)
            if c == 0:
                ax.set_ylabel(f"{lname}\n{tuple(fmap.shape)}", rotation=0, labelpad=55, va="center", fontsize=9)
    fig.suptitle(title, fontsize=13); plt.tight_layout()

def show_filters(model, title):
    w = model.image.backbone.conv1.weight.detach().float().cpu()
    fig, axes = plt.subplots(4, 16, figsize=(13, 3.6))
    for i, ax in enumerate(axes.ravel()):
        f = w[i]; f = (f - f.min()) / (f.max() - f.min() + 1e-8)
        ax.imshow(f.permute(1, 2, 0), interpolation="nearest"); ax.axis("off")
    fig.suptitle(title); plt.tight_layout()

@torch.no_grad()
def show_text_attention(model, sentences, layer=-1, title=""):
    model.eval()
    fig, axes = plt.subplots(len(sentences), TXT_HEADS + 1, figsize=(3.2 * (TXT_HEADS + 1), 3.3 * len(sentences)), squeeze=False)
    for r, s in enumerate(sentences):
        ids = encode(s)[:MAX_LEN].unsqueeze(0).to(DEVICE)
        model.text(ids, keep_attn=True)
        w = model.text.blocks[layer].last_attn[0].float().cpu()                # heads x L x L
        toks = [itos[i] for i in ids[0].tolist()]
        for h in range(TXT_HEADS):
            ax = axes[r, h]; ax.imshow(w[h], cmap="viridis", vmin=0, vmax=1)
            ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=90, fontsize=7)
            ax.set_yticks(range(len(toks))); ax.set_yticklabels(toks if h == 0 else [], fontsize=7)
            ax.set_title(f"head {h}  (row = query, col = key)", fontsize=8)
        ax = axes[r, -1]
        ax.bar(range(len(toks)), w[:, 0, :].mean(0)); ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=90, fontsize=7)
        ax.set_title("what [CLS] reads\n(avg over heads)", fontsize=8)
    fig.suptitle(title, fontsize=12); plt.tight_layout()

# ---------------- BEFORE-training snapshot ----------------
init_state = copy.deepcopy(model.state_dict())
snaps = {}
metrics0, ie0, te0, _, _ = evaluate_model(model, "val")
snaps[0] = make_snapshot("val", ie0, te0)
print("Before training (val):", {k: round(v, 3) for k, v in metrics0.items()})
print(f"Chance level: R@1 image->text ~ {5/len(EV['val']['texts']):.4f}, R@1 text->image ~ {1/len(EV['val']['imgs']):.4f}")

s0 = snaps[0]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ci, ct = reduce_joint(s0["img"], s0["txt"], "pca", key="before")
scatter_joint(ax[0], ci, ct, s0["cat"], "modality", 40, "PCA of the untrained space\n(gray line = a matched image/caption pair)")
plot_simmat(ax[1], s0["img"], s0["txt"], 40, "Similarity matrix, 40 pairs\n(a good model has a red diagonal)")
plot_posneg(ax[2], s0["img"], s0["txt"], "Matched vs mismatched similarity")
plt.suptitle("BEFORE contrastive training", fontsize=13); plt.tight_layout(); save_fig("07_before_embedding"); plt.show()

# raw embedding vectors of 6 matched pairs
fig, ax = plt.subplots(6, 1, figsize=(12, 5.5))
for k in range(6):
    v = torch.stack([s0["img"][k], s0["txt"][k]]).numpy()
    ax[k].imshow(v, aspect="auto", cmap="coolwarm", vmin=-0.25, vmax=0.25); ax[k].set_yticks([0, 1]); ax[k].set_yticklabels(["img", "txt"], fontsize=7)
    ax[k].set_xticks([]); ax[k].set_title(textwrap.shorten(s0["caps"][k], 90) + f"   cos={float(s0['img'][k] @ s0['txt'][k]):.2f}", fontsize=8, loc="left")
plt.suptitle(f"BEFORE: {EMBED_DIM}-number embedding vectors of matched pairs (should look alike after training)"); plt.tight_layout(); save_fig("08_before_vectors"); plt.show()

# feature maps / filters / attention of the untrained model
SAMPLE_IMG = test_tf(EV["test"]["imgs"][int(np.where(EV["test"]["cats"] == "dog")[0][0]) if (EV["test"]["cats"] == "dog").any() else 0])
show_feature_maps(model, SAMPLE_IMG, "ResNet-18 feature maps BEFORE contrastive training" + (" (ImageNet-pretrained backbone)" if PRETRAINED else " (random)"))
save_fig("09_feature_maps_before"); plt.show()
show_filters(model, "First-layer 7x7 filters BEFORE training" + (" (ImageNet)" if PRETRAINED else " (random)"))
save_fig("10_filters_before"); plt.show()
ATT_SENTENCES = [EV["test"]["texts"][0], EV["test"]["texts"][int(EV["test"]["first"][min(5, len(EV["test"]["first"]) - 1)])]]
show_text_attention(model, ATT_SENTENCES, title="Text-encoder attention, last block, BEFORE training (random)")
save_fig("11_attention_before"); plt.show()

## 6. Training
- Loss: `scale * cos(image_i, text_j)` gives an N×N matrix → softmax per row → −log(probability of the correct caption) → same for columns → average.
- AdamW, 1 epoch of warm-up then cosine decay. With a pretrained backbone, the CNN gets a 10× smaller learning rate than the new layers.
- Each epoch we embed the val set to log Recall@K, the learned temperature, and the average similarity of matched vs mismatched pairs, and we save an embedding snapshot at `SNAPSHOT_EPOCHS`.

In [ ]:
backbone_params = list(model.image.backbone.parameters())
bb_ids = {id(p) for p in backbone_params}
other = [p for p in model.parameters() if id(p) not in bb_ids and p is not model.logit_scale]
def split_decay(params):
    return [p for p in params if p.dim() >= 2], [p for p in params if p.dim() < 2]       # no weight decay on biases / norms / temperature
bb_d, bb_n = split_decay(backbone_params); ot_d, ot_n = split_decay(other)
bb_lr = LR * 0.1 if PRETRAINED else LR
optimizer = torch.optim.AdamW([
    {"params": bb_d, "lr": bb_lr, "weight_decay": WEIGHT_DECAY}, {"params": bb_n, "lr": bb_lr, "weight_decay": 0.0},
    {"params": ot_d, "lr": LR, "weight_decay": WEIGHT_DECAY},    {"params": ot_n, "lr": LR, "weight_decay": 0.0},
    {"params": [model.logit_scale], "lr": 2e-3, "weight_decay": 0.0}])      # the temperature gets a bigger LR: it is ONE number and moves slowly otherwise
steps_per_epoch = len(train_loader); total_steps = EPOCHS * steps_per_epoch; warm = steps_per_epoch
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

hist = {k: [] for k in ["epoch", "train_loss", "val_loss", "lr", "temperature", "pos_cos", "neg_cos", "rsum", "epoch_time"] +
        [f"{d}_R@{k}" for d in ("i2t", "t2i") for k in (1, 5, 10)]}
def log_epoch(ep, tr_loss, m, ie, te, lr, t):
    pos, neg = pos_neg(ie, te[EV["val"]["first"]])        # image i vs its first caption
    scale = model.logit_scale.exp().item()
    hist["epoch"].append(ep); hist["train_loss"].append(tr_loss); hist["val_loss"].append(val_loss_from_emb(ie, te, scale))
    hist["lr"].append(lr); hist["temperature"].append(1 / scale); hist["pos_cos"].append(float(pos.mean())); hist["neg_cos"].append(float(neg.mean()))
    hist["rsum"].append(m["rsum"]); hist["epoch_time"].append(t)
    for d in ("i2t", "t2i"):
        for k in (1, 5, 10):
            hist[f"{d}_R@{k}"].append(m[f"{d}_R@{k}"])

log_epoch(0, np.nan, metrics0, ie0, te0, 0.0, 0.0)          # epoch 0 = the untrained model
best_rsum, best_state = -1, None

for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run, n = 0.0, 0
    for imgs, ids, _ in train_loader:
        imgs, ids = imgs.to(DEVICE, non_blocking=True), ids.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            ie, te, scale = model(imgs, ids)
        loss, _ = clip_loss(ie, te, scale)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        with torch.no_grad():
            model.logit_scale.clamp_(0, math.log(100))        # keep temperature >= 0.01
        run += loss.item() * len(imgs); n += len(imgs)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    dt = time.time() - t0
    m, ie_v, te_v, _, _ = evaluate_model(model, "val")
    log_epoch(epoch, run / n, m, ie_v, te_v, optimizer.param_groups[2]["lr"], dt)
    if epoch in SNAPSHOT_EPOCHS:
        snaps[epoch] = make_snapshot("val", ie_v, te_v)
    if m["rsum"] > best_rsum:
        best_rsum, best_state = m["rsum"], copy.deepcopy(model.state_dict())
    print(f"epoch {epoch:2d}/{EPOCHS} | train loss {run/n:.3f} | val loss {hist['val_loss'][-1]:.3f} | "
          f"val R@1 i2t {m['i2t_R@1']:.3f} t2i {m['t2i_R@1']:.3f} | R@10 i2t {m['i2t_R@10']:.3f} | temp {hist['temperature'][-1]:.3f} | {dt:.0f}s")

model.load_state_dict(best_state)
print(f"\nbest val rsum: {best_rsum:.3f} | total train time {sum(hist['epoch_time'])/60:.1f} min")

## 7. Training curves

In [ ]:
ep = np.array(hist["epoch"])
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
ax[0, 0].plot(ep, hist["train_loss"], "o--", label="train"); ax[0, 0].plot(ep, hist["val_loss"], "o-", label="val")
ax[0, 0].axhline(math.log(BATCH_SIZE), color="gray", ls=":", label=f"chance = ln({BATCH_SIZE})"); ax[0, 0].set_title("Contrastive loss  (−log softmax)"); ax[0, 0].legend()
for k, c in zip((1, 5, 10), ("tab:red", "tab:orange", "tab:green")):
    ax[0, 1].plot(ep, hist[f"i2t_R@{k}"], "o-", color=c, label=f"R@{k}")
    ax[0, 2].plot(ep, hist[f"t2i_R@{k}"], "o-", color=c, label=f"R@{k}")
ax[0, 1].set_title("Image → text retrieval (val)"); ax[0, 2].set_title("Text → image retrieval (val)")
for a in ax[0, 1:]:
    a.legend(); a.set_ylim(0, 1)
ax[1, 0].plot(ep, hist["pos_cos"], "o-", color="tab:green", label="matched pairs"); ax[1, 0].plot(ep, hist["neg_cos"], "o-", color="tab:gray", label="mismatched pairs")
ax[1, 0].set_title("Mean cosine similarity (val)"); ax[1, 0].legend()
ax[1, 1].plot(ep, hist["temperature"], "o-", color="tab:purple"); ax[1, 1].axhline(INIT_TEMP, color="gray", ls=":"); ax[1, 1].set_title("Learned temperature τ  (dotted = start)")
ax[1, 2].plot(ep[1:], hist["lr"][1:], "o-", color="tab:brown"); ax[1, 2].set_title("Learning rate (head, end of epoch)")
for a in ax.ravel():
    a.set_xlabel("epoch"); a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.suptitle("Training dynamics (epoch 0 = untrained)", fontsize=14); plt.tight_layout(); save_fig("12_training_curves"); plt.show()

# Matched-pair vs mismatched-pair similarity distributions through training
eps_s = sorted(snaps)
fig, ax = plt.subplots(1, len(eps_s), figsize=(3.4 * len(eps_s), 3.2), sharey=True)
for a, e in zip(np.atleast_1d(ax), eps_s):
    pos, neg = pos_neg(snaps[e]["img"], snaps[e]["txt"])
    a.hist(neg, bins=50, density=True, alpha=0.6, color="tab:gray"); a.hist(pos, bins=30, density=True, alpha=0.7, color="tab:green"); a.set_title(f"epoch {e}", fontsize=10); a.set_xlim(-1, 1)
plt.suptitle("Green = matched pairs, gray = mismatched: they separate as training proceeds"); plt.tight_layout(); save_fig("13_posneg_over_epochs"); plt.show()

## 8. Test-set retrieval: untrained vs trained
Standard protocol: every test image is ranked against **all** test captions (image→text, correct if *any* of its 5 captions is in the top K) and every caption against all test images (text→image). `medr` = median rank of the first correct answer (1 is perfect).

In [ ]:
# after training (best epoch)
res_after, ie_a, te_a, rank_i2t, rank_t2i = evaluate_model(model, "test")
# before training (restore the initial weights, evaluate, then put the best weights back)
model.load_state_dict(init_state); res_before, ie_b, te_b, rank_i2t_b, rank_t2i_b = evaluate_model(model, "test"); model.load_state_dict(best_state)
tsnap = {"before": make_snapshot("test", ie_b, te_b), "after": make_snapshot("test", ie_a, te_a)}

n_img, n_txt = len(EV["test"]["imgs"]), len(EV["test"]["texts"])
print(f"TEST: {n_img} images x {n_txt} captions")
print(f"{'':18s}{'before':>10s}{'after':>10s}")
for k in ["i2t_R@1", "i2t_R@5", "i2t_R@10", "i2t_medr", "t2i_R@1", "t2i_R@5", "t2i_R@10", "t2i_medr", "rsum"]:
    print(f"{k:18s}{res_before[k]:>10.3f}{res_after[k]:>10.3f}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
x, w = np.arange(3), 0.38
for a, d, name in [(ax[0], "i2t", "Image → text"), (ax[1], "t2i", "Text → image")]:
    b1 = a.bar(x - w/2, [res_before[f"{d}_R@{k}"] for k in (1, 5, 10)], w, label="before", color="tab:gray")
    b2 = a.bar(x + w/2, [res_after[f"{d}_R@{k}"] for k in (1, 5, 10)], w, label="after", color="tab:green")
    a.bar_label(b1, fmt="%.3f", fontsize=8); a.bar_label(b2, fmt="%.3f", fontsize=8)
    a.set_xticks(x); a.set_xticklabels(["R@1", "R@5", "R@10"]); a.set_ylim(0, 1.05); a.set_title(name + " (test)"); a.legend()
for r, c, l in [(rank_i2t_b, "tab:gray", "before"), (rank_i2t, "tab:blue", "after i2t"), (rank_t2i, "tab:orange", "after t2i")]:
    ax[2].hist(r.numpy() + 1, bins=np.logspace(0, math.log10(max(n_img, n_txt)), 40), alpha=0.6, color=c, label=l)
ax[2].set_xscale("log"); ax[2].set_title("Rank of the correct answer (log scale, left = better)"); ax[2].set_xlabel("rank"); ax[2].legend()
plt.tight_layout(); save_fig("14_test_retrieval"); plt.show()

# R@1 per theme
fig, ax = plt.subplots(figsize=(13, 4.2))
cats_t = EV["test"]["cats"]; t2i_t = EV["test"]["t2i"].numpy()
present = [c for c in CAT_NAMES if (cats_t == c).sum() >= 3]
r_i = [(rank_i2t.numpy()[cats_t == c] < 1).mean() for c in present]
r_t = [(rank_t2i.numpy()[cats_t[t2i_t] == c] < 1).mean() for c in present]
xx = np.arange(len(present))
ax.bar(xx - 0.2, r_i, 0.4, label="image → text R@1", color="tab:blue"); ax.bar(xx + 0.2, r_t, 0.4, label="text → image R@1", color="tab:orange")
ax.set_xticks(xx); ax.set_xticklabels([f"{c}\n(n={(cats_t==c).sum()})" for c in present], fontsize=8); ax.set_title("Which kinds of photos are easy? (R@1 by keyword theme)"); ax.legend()
plt.tight_layout(); save_fig("15_per_theme_recall"); plt.show()

## 9. Similarity structure (where the −log softmax comes from)
Left: cosine similarity of 50 test images vs their own caption. Middle: after dividing by the learned temperature and applying softmax, **row i** is the probability the model gives to each caption for image i. The loss for that row is −log of the diagonal entry. Right: that probability for the first 50 images.

In [ ]:
sb, sa = tsnap["before"], tsnap["after"]
n = min(50, len(sb["img"]))
scale_now = model.logit_scale.exp().item()
fig, ax = plt.subplots(2, 4, figsize=(20, 9))
for r, (s, lab) in enumerate([(sb, "BEFORE"), (sa, "AFTER")]):
    im = plot_simmat(ax[r, 0], s["img"], s["txt"], n, f"{lab}: cosine similarity")
    plt.colorbar(im, ax=ax[r, 0], fraction=0.046)
    sc = 1 / INIT_TEMP if r == 0 else scale_now
    P = torch.softmax(sc * (s["img"][:n] @ s["txt"][:n].T), dim=1).numpy()
    im = ax[r, 1].imshow(P, cmap="magma", vmin=0, vmax=1); plt.colorbar(im, ax=ax[r, 1], fraction=0.046)
    ax[r, 1].set_title(f"{lab}: softmax(sim / τ), τ={1/sc:.3f}\n(row i = p(caption j | image i))", fontsize=10); ax[r, 1].set_xlabel("caption j"); ax[r, 1].set_ylabel("image i")
    p_corr = np.diag(P)
    ax[r, 2].bar(range(n), p_corr, color="tab:green"); ax[r, 2].axhline(1 / n, color="red", ls="--", label=f"chance 1/{n}"); ax[r, 2].set_ylim(0, 1)
    ax[r, 2].set_title(f"{lab}: p(correct caption) per image  (mean {p_corr.mean():.2f})", fontsize=10); ax[r, 2].legend(); ax[r, 2].set_xlabel("image i")
    ax[r, 3].hist(-np.log(p_corr + 1e-12), bins=25, color="tab:red"); ax[r, 3].set_title(f"{lab}: per-image loss  −log p(correct)  (mean {(-np.log(p_corr+1e-12)).mean():.2f})", fontsize=10)
plt.suptitle("From similarity → probability → loss", fontsize=14); plt.tight_layout(); save_fig("16_similarity_to_loss"); plt.show()

# Effect of the temperature on the SAME similarity matrix
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
S = (sa["img"][:n] @ sa["txt"][:n].T)
for a, tau in zip(ax, [1.0, 0.2, INIT_TEMP, 1 / scale_now]):
    P = torch.softmax(S / tau, 1).numpy(); a.imshow(P, cmap="magma", vmin=0, vmax=1)
    a.set_title(f"τ = {tau:.3f}\nmean p(correct) = {np.diag(P).mean():.2f}", fontsize=10)
plt.suptitle("Same trained similarities, different temperature: τ controls how sharp the softmax is"); plt.tight_layout(); save_fig("17_temperature_effect"); plt.show()

## 10. Retrieval galleries (the model in action)
Green border = a correct match. `search("your own sentence")` works with any text.

In [ ]:
test_imgs, test_texts = EV["test"]["imgs"], EV["test"]["texts"]
def show_text_to_images(query_idx_list, k=5):
    fig, axes = plt.subplots(len(query_idx_list), k, figsize=(3 * k, 3.3 * len(query_idx_list)), squeeze=False)
    for r, qi in enumerate(query_idx_list):
        top = (te_a[qi] @ ie_a.T).topk(k).indices.tolist(); gt = EV["test"]["t2i"][qi].item()
        for c, j in enumerate(top):
            ax = axes[r, c]; ax.imshow(test_imgs[j]); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor("green" if j == gt else "red"); sp.set_linewidth(4 if j == gt else 1)
            ax.set_title(f"#{c+1}  cos={float(te_a[qi] @ ie_a[j]):.2f}", fontsize=9)
        axes[r, 0].set_ylabel(textwrap.fill(test_texts[qi], 24), fontsize=9, rotation=0, labelpad=75, va="center")
    plt.suptitle("TEXT → IMAGE: query caption on the left, top-5 photos", fontsize=13); plt.tight_layout()

def show_image_to_texts(img_idx_list, k=3):
    fig, axes = plt.subplots(len(img_idx_list), 2, figsize=(11, 2.6 * len(img_idx_list)), gridspec_kw=dict(width_ratios=[1, 3]), squeeze=False)
    for r, i in enumerate(img_idx_list):
        top = (ie_a[i] @ te_a.T).topk(k).indices.tolist()
        axes[r, 0].imshow(test_imgs[i]); axes[r, 0].axis("off")
        lines = [("✓ " if EV["test"]["t2i"][j].item() == i else "✗ ") + f"{float(ie_a[i] @ te_a[j]):.2f}  " + textwrap.shorten(test_texts[j], 75) for j in top]
        axes[r, 1].axis("off"); axes[r, 1].text(0, 0.5, "\n".join(lines), va="center", fontsize=10)
    plt.suptitle("IMAGE → TEXT: top-3 captions (✓ = one of its own 5 captions)", fontsize=13); plt.tight_layout()

rs = np.random.RandomState(1)
show_text_to_images(rs.choice(n_txt, 4, replace=False).tolist()); save_fig("18_text_to_image"); plt.show()
show_image_to_texts(rs.choice(n_img, 5, replace=False).tolist()); save_fig("19_image_to_text"); plt.show()

@torch.no_grad()
def search(query, k=5):
    q = embed_texts(model, [query])[0]
    sims = q @ ie_a.T; top = sims.topk(k).indices.tolist()
    fig, axes = plt.subplots(1, k, figsize=(3 * k, 3.4))
    for a, j in zip(np.atleast_1d(axes), top):
        a.imshow(test_imgs[j]); a.axis("off"); a.set_title(f"cos={sims[j]:.2f}", fontsize=9)
    plt.suptitle(f'"{query}"'); plt.tight_layout(); plt.show()

search("a dog running through the grass")
search("a person skiing down a snowy mountain")
search("children playing in the water")

## 11. Embedding space in **2D**
The 512→256 vectors live on a 256-D hypersphere, so we squash them to 2D three different ways. **PCA** keeps the directions of largest variance (global structure, linear). **t-SNE** and **UMAP** preserve local neighbourhoods (clusters). Image and caption embeddings are projected **together** so they share axes. Circles = images, triangles = captions, gray lines connect a photo to its own caption.

In [ ]:
# (a) how the space organizes itself during training (PCA of the val snapshots)
eps_s = sorted(snaps)
fig, axes = plt.subplots(2, len(eps_s), figsize=(3.4 * len(eps_s), 7), squeeze=False)
for k, e in enumerate(eps_s):
    ci, ct = reduce_joint(snaps[e]["img"], snaps[e]["txt"], "pca", key=f"val{e}")
    scatter_joint(axes[0, k], ci, ct, snaps[e]["cat"], "modality", 40, f"epoch {e}")
    scatter_joint(axes[1, k], ci, ct, snaps[e]["cat"], "category", 0, "")
cat_legend(axes[1, -1], snaps[eps_s[-1]]["cat"], "lower right")
axes[0, 0].set_ylabel("colored by modality", fontsize=11); axes[1, 0].set_ylabel("colored by theme", fontsize=11)
plt.suptitle("Embedding space during training (PCA, val subset)", fontsize=14); plt.tight_layout(); save_fig("20_space_evolution_pca"); plt.show()

# (b) before vs after, three projection methods (test subset)
methods = [m for m in ("pca", "tsne", "umap") if m != "umap" or HAS_UMAP]
fig, axes = plt.subplots(3, len(methods), figsize=(5.6 * len(methods), 16), squeeze=False)
for c, m in enumerate(methods):
    cb, tb = reduce_joint(tsnap["before"]["img"], tsnap["before"]["txt"], m, key="tb")
    ca, ta = reduce_joint(tsnap["after"]["img"], tsnap["after"]["txt"], m, key="ta")
    scatter_joint(axes[0, c], cb, tb, tsnap["before"]["cat"], "modality", 40, f"{m.upper()}  BEFORE training")
    scatter_joint(axes[1, c], ca, ta, tsnap["after"]["cat"], "modality", 40, f"{m.upper()}  AFTER training")
    scatter_joint(axes[2, c], ca, ta, tsnap["after"]["cat"], "category", 0, f"{m.upper()}  AFTER, colored by theme")
cat_legend(axes[2, 0], tsnap["after"]["cat"], "best")
plt.suptitle("Test subset: image + caption embeddings in a shared 2D map", fontsize=14); plt.tight_layout(); save_fig("21_space_2d_methods"); plt.show()

# (c) each modality on its own (t-SNE), colored by theme: does each encoder organize by meaning by itself?
sa_ = tsnap["after"]
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, key, name in [(ax[0], "img", "IMAGE embeddings only"), (ax[1], "txt", "CAPTION embeddings only")]:
    Z = TSNE(2, perplexity=min(30, (len(sa_[key]) - 1) / 3), init="pca", learning_rate="auto", random_state=SEED).fit_transform(sa_[key].numpy())
    a.scatter(Z[:, 0], Z[:, 1], s=16, c=np.array([CAT_COLOR[c] for c in sa_["cat"]]), alpha=0.85); a.set_title(name + " (t-SNE, after training)"); a.set_xticks([]); a.set_yticks([])
cat_legend(ax[1], sa_["cat"], "best")
plt.tight_layout(); save_fig("22_space_per_modality"); plt.show()

# (d) the same map, but each point shows its caption (a readable neighbourhood)
ca, ta = reduce_joint(sa_["img"], sa_["txt"], "tsne", key="ta")
fig, ax = plt.subplots(figsize=(13, 10))
scatter_joint(ax, ca, ta, sa_["cat"], "category", 0, "t-SNE of captions (triangles) with 45 labels")
for i in np.random.RandomState(3).choice(len(ta), min(45, len(ta)), replace=False):
    ax.annotate(textwrap.shorten(sa_["caps"][i], 38, placeholder="…"), ta[i], fontsize=6.5, alpha=0.85)
cat_legend(ax, sa_["cat"], "lower left"); plt.tight_layout(); save_fig("23_space_labeled"); plt.show()

## 12. Embedding space in **3D**
Four views: (a) static 3D PCA from several angles, (b) the points projected onto the unit sphere (what a normalized embedding really is), (c) **interactive** plotly maps you can rotate / zoom / hover for the caption (PCA, t-SNE, UMAP) saved as HTML, (d) an **animation** of the space forming during training.

In [ ]:
def scatter3d_mpl(ax, ci, ct, cats_, mode, n_lines=40):
    if n_lines:
        for i in np.random.RandomState(0).choice(len(ci), min(n_lines, len(ci)), replace=False):
            ax.plot([ci[i, 0], ct[i, 0]], [ci[i, 1], ct[i, 1]], [ci[i, 2], ct[i, 2]], color="gray", lw=0.4, alpha=0.6)
    if mode == "modality":
        ax.scatter(*ci.T, s=8, c=COLORS["image"], marker="o", alpha=0.7); ax.scatter(*ct.T, s=14, c=COLORS["text"], marker="^", alpha=0.7)
    else:
        col = np.array([CAT_COLOR[c] for c in cats_]); ax.scatter(*ci.T, s=8, c=col, marker="o", alpha=0.8); ax.scatter(*ct.T, s=14, c=col, marker="^", alpha=0.8)

# (a) static 3D PCA, 3 viewing angles x (before, after-by-modality, after-by-theme)
cb3, tb3 = reduce_joint(tsnap["before"]["img"], tsnap["before"]["txt"], "pca", 3, key="tb")
ca3, ta3 = reduce_joint(tsnap["after"]["img"], tsnap["after"]["txt"], "pca", 3, key="ta")
fig = plt.figure(figsize=(18, 10))
rows = [("BEFORE (modality)", cb3, tb3, "before", "modality"), ("AFTER (modality)", ca3, ta3, "after", "modality"), ("AFTER (theme)", ca3, ta3, "after", "category")]
for r, (lab, ci, ct, key, mode) in enumerate(rows):
    for c, azim in enumerate((20, 110, 200)):
        ax = fig.add_subplot(3, 3, r * 3 + c + 1, projection="3d")
        scatter3d_mpl(ax, ci, ct, tsnap[key]["cat"], mode); ax.view_init(elev=22, azim=azim)
        ax.set_title(f"{lab}, azim {azim}", fontsize=9); ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
plt.suptitle("3D PCA of the test subset, seen from three angles", fontsize=14); plt.tight_layout(); save_fig("24_space_3d_static"); plt.show()

# (b) the unit-sphere picture: keep the top-3 PCA directions and re-normalize to length 1
def sphere_ax(ax, ci, ct, title):
    u, v = np.mgrid[0:2 * np.pi:40j, 0:np.pi:20j]
    ax.plot_wireframe(np.cos(u) * np.sin(v), np.sin(u) * np.sin(v), np.cos(v), color="lightgray", lw=0.3, alpha=0.6)
    nz = lambda z: z / (np.linalg.norm(z, axis=1, keepdims=True) + 1e-9)
    ax.scatter(*nz(ci).T, s=8, c=COLORS["image"], alpha=0.7); ax.scatter(*nz(ct).T, s=14, c=COLORS["text"], marker="^", alpha=0.7)
    ax.set_box_aspect((1, 1, 1)); ax.set_title(title, fontsize=10); ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
fig = plt.figure(figsize=(15, 7))
sphere_ax(fig.add_subplot(1, 2, 1, projection="3d"), cb3, tb3, "BEFORE: images (blue) and captions (orange) are on different parts of the sphere")
sphere_ax(fig.add_subplot(1, 2, 2, projection="3d"), ca3, ta3, "AFTER: they overlap")
plt.suptitle("Embeddings are unit vectors = points on a hypersphere (shown here: top-3 PCA directions, re-normalized)"); plt.tight_layout(); save_fig("25_sphere"); plt.show()

# (c) interactive plotly maps
def plotly_3d(snap, method, title, fname, key):
    ci, ct = reduce_joint(snap["img"], snap["txt"], method, 3, key=key)
    fig = go.Figure()
    xs, ys, zs = [], [], []
    for i in np.random.RandomState(0).choice(len(ci), min(60, len(ci)), replace=False):
        xs += [ci[i, 0], ct[i, 0], None]; ys += [ci[i, 1], ct[i, 1], None]; zs += [ci[i, 2], ct[i, 2], None]
    fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color="gray", width=1), name="matched pair", hoverinfo="skip", opacity=0.5))
    for modality, Z, sym, size in (("image", ci, "circle", 3), ("text", ct, "diamond", 4)):
        for c in CAT_NAMES:
            m = snap["cat"] == c
            if m.sum() == 0:
                continue
            fig.add_trace(go.Scatter3d(x=Z[m, 0], y=Z[m, 1], z=Z[m, 2], mode="markers", name=f"{modality} | {c}", legendgroup=c,
                                       marker=dict(size=size, color=to_hex(CAT_COLOR[c]), symbol=sym, opacity=0.85),
                                       text=[snap["caps"][i] for i in np.where(m)[0]], hovertemplate="%{text}<extra>" + f"{modality} | {c}" + "</extra>"))
    fig.update_layout(title=title, height=720, margin=dict(l=0, r=0, t=40, b=0), legend=dict(itemsizing="constant", font=dict(size=9)))
    fig.write_html(f"images/{fname}.html"); fig.show()

plotly_3d(tsnap["after"], "pca", "AFTER training: 3D PCA (rotate, zoom, hover = caption)", "26_interactive_pca", "ta")
plotly_3d(tsnap["after"], "tsne", "AFTER training: 3D t-SNE", "27_interactive_tsne", "ta")
if HAS_UMAP:
    plotly_3d(tsnap["after"], "umap", "AFTER training: 3D UMAP", "28_interactive_umap", "ta")
plotly_3d(tsnap["before"], "pca", "BEFORE training: 3D PCA (compare with the one above)", "29_interactive_pca_before", "tb")

# (d) animation of the space forming during training (val snapshots, PCA-3D, each frame rotated to match the last one)
def training_animation(snaps, fname):
    eps_ = sorted(snaps); Z = {e: reduce_joint(snaps[e]["img"], snaps[e]["txt"], "pca", 3, key=f"val{e}") for e in eps_}
    ref = np.concatenate(Z[eps_[-1]], 0); frames_xyz = {}
    for e in eps_:
        cur = np.concatenate(Z[e], 0); R, _ = orthogonal_procrustes(cur - cur.mean(0), ref - ref.mean(0))     # rotate so frames don't spin randomly
        aligned = (cur - cur.mean(0)) @ R; frames_xyz[e] = (aligned[:len(Z[e][0])], aligned[len(Z[e][0]):])
    lim = max(np.abs(np.concatenate([np.concatenate(v) for v in frames_xyz.values()])).max(), 1e-6) * 1.05
    def tr(e):
        ci, ct = frames_xyz[e]
        return [go.Scatter3d(x=ci[:, 0], y=ci[:, 1], z=ci[:, 2], mode="markers", name="image", marker=dict(size=3, color=PL_COLORS["image"], opacity=0.8)),
                go.Scatter3d(x=ct[:, 0], y=ct[:, 1], z=ct[:, 2], mode="markers", name="text", marker=dict(size=4, color=PL_COLORS["text"], symbol="diamond", opacity=0.8))]
    fig = go.Figure(data=tr(eps_[0]), frames=[go.Frame(data=tr(e), name=str(e)) for e in eps_])
    ax_ = dict(range=[-lim, lim], showticklabels=False)
    fig.update_layout(height=720, title="Embedding space during training: press Play or drag the slider (epoch 0 = untrained)", scene=dict(xaxis=ax_, yaxis=ax_, zaxis=ax_, aspectmode="cube"),
        updatemenus=[dict(type="buttons", showactive=False, buttons=[dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=900, redraw=True), fromcurrent=True)])])],
        sliders=[dict(steps=[dict(method="animate", label=f"epoch {e}", args=[[str(e)], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for e in eps_])])
    fig.write_html(f"images/{fname}.html"); fig.show()
training_animation(snaps, "30_training_animation")

## 13. Embedding diagnostics (numbers that describe the space)
- **Modality gap**: distance between the image centroid and the text centroid. Contrastive models usually keep a gap even when retrieval works.
- **Alignment** (↓): average squared distance between matched pairs. **Uniformity** (↓): how evenly points spread over the sphere (Wang & Isola, 2020).
- **kNN modality mixing**: for each image, the fraction of its 10 nearest neighbours (in the joint space) that are captions. 0 = two separate islands, 0.5 = fully mixed.
- **Effective rank**: how many of the 256 dimensions are really used. 

In [ ]:
def modality_gap(img, txt):
    mi, mt = img.mean(0), txt.mean(0)
    return (mt - mi).norm().item(), F.cosine_similarity(mi, mt, dim=0).item()
def alignment(img, txt):
    return (img - txt).pow(2).sum(1).mean().item()
def uniformity(z, t=2):
    return torch.log(torch.exp(-t * torch.pdist(z, p=2).pow(2)).mean()).item()
def knn_mix(img, txt, k=10):
    Z = torch.cat([img, txt]); sim = Z @ Z.T; sim.fill_diagonal_(-2)
    idx = sim.topk(k, dim=1).indices; is_txt = torch.arange(len(Z)) >= len(img)
    return is_txt[idx[:len(img)]].float().mean().item()
def eff_rank(Z):
    s = torch.linalg.svdvals(Z - Z.mean(0)); p = s / s.sum()
    return torch.exp(-(p * torch.log(p + 1e-12)).sum()).item()

diag = {}
for key in ("before", "after"):
    s = tsnap[key]; gap, ccos = modality_gap(s["img"], s["txt"])
    diag[key] = dict(gap=gap, centroid_cos=ccos, alignment=alignment(s["img"], s["txt"]), unif_img=uniformity(s["img"]), unif_txt=uniformity(s["txt"]),
                     knn_mix=knn_mix(s["img"], s["txt"]), eff_rank=eff_rank(torch.cat([s["img"], s["txt"]])))
print(f"{'':16s}{'before':>10s}{'after':>10s}")
for k in diag["before"]:
    print(f"{k:16s}{diag['before'][k]:>10.3f}{diag['after'][k]:>10.3f}")

fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for a, k, t in zip(ax, ["gap", "alignment", "knn_mix", "eff_rank"], ["Modality gap (centroid distance)", "Alignment (lower = matched pairs closer)", "kNN modality mixing (0.5 = mixed)", "Effective rank of the space"]):
    bars = a.bar(["before", "after"], [diag["before"][k], diag["after"][k]], color=["tab:gray", "tab:green"]); a.bar_label(bars, fmt="%.3f"); a.set_title(t, fontsize=10)
plt.tight_layout(); save_fig("31_diagnostics_bars"); plt.show()

# (a) the "gap axis" view: x = direction between the two centroids, y = largest remaining variance
def gap_view(ax, img, txt, title):
    Z = torch.cat([img, txt]).numpy(); X = Z - Z.mean(0)
    g = (txt.mean(0) - img.mean(0)).numpy(); u = g / (np.linalg.norm(g) + 1e-9)
    x = X @ u; R = X - np.outer(x, u); y = PCA(1, random_state=SEED).fit_transform(R)[:, 0]
    n_ = len(img); ax.scatter(x[:n_], y[:n_], s=10, c=COLORS["image"], alpha=0.6, label="image"); ax.scatter(x[n_:], y[n_:], s=14, c=COLORS["text"], marker="^", alpha=0.6, label="text")
    ax.set_title(title, fontsize=10); ax.set_xlabel("← image side      text side →  (centroid-difference axis)"); ax.legend(fontsize=8)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
gap_view(ax[0], tsnap["before"]["img"], tsnap["before"]["txt"], "BEFORE"); gap_view(ax[1], tsnap["after"]["img"], tsnap["after"]["txt"], "AFTER: the leftover modality gap")
plt.tight_layout(); save_fig("32_modality_gap"); plt.show()

# (b) spectrum: how many dimensions are used
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for key, c in [("before", "tab:gray"), ("after", "tab:green")]:
    s = tsnap[key]; Z = torch.cat([s["img"], s["txt"]]); sv = torch.linalg.svdvals(Z - Z.mean(0)).numpy(); var = sv ** 2 / (sv ** 2).sum()
    ax[0].semilogy(np.arange(1, len(sv) + 1), sv / sv[0], color=c, label=key); ax[1].plot(np.arange(1, len(var) + 1), np.cumsum(var), color=c, label=key)
ax[0].set_title("Singular values (normalized, log)"); ax[0].set_xlabel("component"); ax[0].legend(); ax[1].set_title("Cumulative variance explained"); ax[1].set_xlabel("# components"); ax[1].legend()
plt.tight_layout(); save_fig("33_spectrum"); plt.show()

# (c) matched pairs: cosine per theme, and the raw vectors
pos_after, neg_after = pos_neg(tsnap["after"]["img"], tsnap["after"]["txt"])
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
dfc = [(c, p) for c, p in zip(tsnap["after"]["cat"], pos_after)]
order_c = [c for c in CAT_NAMES if any(x == c for x, _ in dfc)]
sns.boxplot(x=[c for c, _ in dfc], y=[p for _, p in dfc], order=order_c, ax=ax[0], color="tab:green"); ax[0].axhline(neg_after.mean(), color="gray", ls="--", label="mean mismatched pair"); ax[0].legend()
ax[0].set_title("Cosine of matched image/caption pairs, by theme (after)"); ax[0].tick_params(axis="x", rotation=30)
plot_posneg(ax[1], tsnap["after"]["img"], tsnap["after"]["txt"], "Matched vs mismatched similarity (test, after)")
plt.tight_layout(); save_fig("34_pair_similarity"); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 5.2))
for r, key in enumerate(("before", "after")):
    s = tsnap[key]; sel = np.argsort(-(s["img"] * s["txt"]).sum(1).numpy())[:8] if key == "after" else np.arange(8)
    I, Tt = s["img"][sel].numpy(), s["txt"][sel].numpy(); lim = max(np.abs(I).max(), np.abs(Tt).max())
    for c, (M, t) in enumerate([(I, "image vectors"), (Tt, "caption vectors"), (I * Tt, "element-wise product (its SUM is the cosine)")]):
        v = lim if c < 2 else lim ** 2
        axes[r, c].imshow(M, aspect="auto", cmap="coolwarm", vmin=-v, vmax=v); axes[r, c].set_yticks(range(8)); axes[r, c].set_yticklabels([f"cos={float(s['img'][j] @ s['txt'][j]):.2f}" for j in sel], fontsize=7)
        axes[r, c].set_title(f"{key.upper()}: {t}", fontsize=9); axes[r, c].set_xlabel("embedding dimension")
plt.suptitle("8 matched pairs as raw vectors (after: the 8 best-matched pairs)"); plt.tight_layout(); save_fig("35_vectors_before_after"); plt.show()

# (d) theme centroids: are image-centroid and caption-centroid of the same theme close?
sA = tsnap["after"]; present = [c for c in CAT_NAMES if (sA["cat"] == c).sum() >= 3]
Ci = torch.stack([sA["img"][sA["cat"] == c].mean(0) for c in present]); Ct = torch.stack([sA["txt"][sA["cat"] == c].mean(0) for c in present])
Ci, Ct = F.normalize(Ci, dim=-1), F.normalize(Ct, dim=-1)
fig, ax = plt.subplots(1, 2, figsize=(17, 6.5))
sns.heatmap((Ci @ Ct.T).numpy(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, xticklabels=present, yticklabels=present, ax=ax[0], cbar=False)
ax[0].set_xlabel("caption-centroid of theme"); ax[0].set_ylabel("image-centroid of theme"); ax[0].set_title("Cosine between theme centroids (diagonal should win)")
Cj = torch.cat([Ci, Ct]).numpy(); labs = [f"img:{c}" for c in present] + [f"txt:{c}" for c in present]
hierarchy.dendrogram(hierarchy.linkage(Cj, "average", metric="cosine"), labels=labs, orientation="right", ax=ax[1], leaf_font_size=9)
ax[1].set_title("Hierarchical clustering of theme centroids\n(img:dog should merge with txt:dog first)")
plt.tight_layout(); save_fig("36_theme_centroids"); plt.show()

# (e) k-means on the joint space: does a cluster contain both photos and captions?
K = min(8, len(sA["img"]) // 5); Zj = torch.cat([sA["img"], sA["txt"]]).numpy(); n_ = len(sA["img"])
km = KMeans(K, n_init=10, random_state=SEED).fit(Zj); lab = km.labels_; is_img = np.arange(len(Zj)) < n_
stop = set("a an the in on of with and is are to at his her their its up down while by for from as it two three one".split())
fig, ax = plt.subplots(figsize=(11, 4))
fi = [is_img[lab == k].mean() for k in range(K)]
ax.bar(range(K), fi, color="tab:blue", label="images"); ax.bar(range(K), [1 - f for f in fi], bottom=fi, color="tab:orange", label="captions"); ax.axhline(0.5, color="k", ls="--", lw=0.8)
ax.set_xlabel("k-means cluster"); ax.set_title("Cluster composition: bars near 50/50 = photos and their captions share the cluster"); ax.legend()
plt.tight_layout(); save_fig("37_kmeans_mix"); plt.show()
print("Top words in the CAPTIONS of each cluster:")
for k in range(K):
    words = Counter(w for j in np.where((lab == k) & ~is_img)[0] for w in tokenize(sA["caps"][j - n_]) if w not in stop)
    print(f"  cluster {k} (n={int((lab==k).sum()):3d}, images {fi[k]:.0%}): ", ", ".join(w for w, _ in words.most_common(7)))

## 14. Interpretability: what is the model looking at / reading?
1. **Text attention** (after training): which words does `[CLS]` read in each head?
2. **Grad-CAM for similarity**: which pixels make an image similar to a given sentence? We back-propagate `cos(image, text)` into the last ResNet stage. Same idea as your ResNet notebook's Grad-CAM, but the "class score" is replaced by the similarity to a caption.
3. **Prompt similarity matrix**: does the text encoder understand synonyms and word order?
4. **Zero-shot classification**: use sentences as class prototypes, no classifier was trained.

In [ ]:
# (1) attention after training
show_text_attention(model, ATT_SENTENCES, title="Text-encoder attention, last block, AFTER training")
save_fig("38_attention_after"); plt.show()

# (feature maps / filters after training)
show_feature_maps(model, SAMPLE_IMG, "ResNet-18 feature maps AFTER contrastive training"); save_fig("39_feature_maps_after"); plt.show()
show_filters(model, "First-layer filters AFTER training"); save_fig("40_filters_after"); plt.show()

# (2) Grad-CAM for the similarity between an image and a text
def grad_cam_sim(model, img_tensor, text_vec, layer):
    model.eval(); store = {}
    def hook(_, __, out):
        store["act"] = out; out.register_hook(lambda g: store.__setitem__("grad", g))
    h = layer.register_forward_hook(hook)
    with torch.enable_grad():
        z = model.image(img_tensor.unsqueeze(0).to(DEVICE)); h.remove()
        sim = (z * text_vec.to(DEVICE)[None]).sum(); model.zero_grad(); sim.backward()
    act, grad = store["act"][0], store["grad"][0]
    cam = F.relu((grad.mean(dim=(1, 2))[:, None, None] * act).sum(0))
    cam = F.interpolate(cam[None, None], size=img_tensor.shape[1:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy(), sim.item()

L4 = model.image.backbone.layer4
# one image per row, its OWN caption as the query
idxs = np.random.RandomState(5).choice(n_img, 6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(17, 6.2))
for k, i in enumerate(idxs):
    x = test_tf(test_imgs[i]); q = te_a[int(EV["test"]["first"][i])]
    cam, s_ = grad_cam_sim(model, x, q, L4)
    axes[0, k].imshow(denorm(x)); axes[0, k].set_title(textwrap.fill(test_texts[int(EV['test']['first'][i])], 28), fontsize=8)
    axes[1, k].imshow(denorm(x)); axes[1, k].imshow(cam, cmap="jet", alpha=0.45); axes[1, k].set_title(f"cos = {s_:.2f}", fontsize=9)
for a in axes.ravel():
    a.axis("off")
plt.suptitle("Grad-CAM: which regions make the photo match ITS OWN caption", fontsize=13); plt.tight_layout(); save_fig("41_gradcam_own_caption"); plt.show()

# one image, many different queries: the heat-map should move with the words
QUERIES = ["a dog", "a person", "grass", "water", "snow", "a bike", "the sky"]
i = int(idxs[0]); x = test_tf(test_imgs[i]); QV = embed_texts(model, QUERIES)
fig, axes = plt.subplots(1, len(QUERIES) + 1, figsize=(2.6 * (len(QUERIES) + 1), 3.4))
axes[0].imshow(denorm(x)); axes[0].set_title("photo"); axes[0].axis("off")
for k, q in enumerate(QUERIES):
    cam, s_ = grad_cam_sim(model, x, QV[k], L4)
    axes[k + 1].imshow(denorm(x)); axes[k + 1].imshow(cam, cmap="jet", alpha=0.45); axes[k + 1].set_title(f'"{q}"\ncos={s_:.2f}', fontsize=9); axes[k + 1].axis("off")
plt.suptitle("Same photo, different text query"); plt.tight_layout(); save_fig("42_gradcam_queries"); plt.show()

# (3) prompt similarity matrix: synonyms and word order
PROMPTS = ["a dog runs on the grass", "a dog running through a field", "a man rides a bike", "a person on a bicycle",
           "a child plays in the snow", "kids playing in the snow", "a girl swims in the water", "people at the beach",
           "a dog bites a man", "a man bites a dog"]
PE = embed_texts(model, PROMPTS)
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap((PE @ PE.T).numpy(), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, xticklabels=PROMPTS, yticklabels=PROMPTS, ax=ax, cbar=False)
ax.tick_params(axis="x", rotation=60); ax.set_title("Text-to-text cosine: synonyms should be close; 'dog bites man' vs 'man bites dog' shows word-order sensitivity")
plt.tight_layout(); save_fig("43_prompt_similarity"); plt.show()

# (4) zero-shot classification with sentences as prototypes (labels come from the noisy keyword themes, so treat this as a sanity check)
ZS = OrderedDict([("dog", "a dog"), ("water", "people swimming in the water at the beach"), ("snow", "a person skiing in the snow"),
                  ("ball sport", "people playing a ball game"), ("vehicle", "a person riding a bike"), ("climbing", "a person climbing a rock on a mountain"),
                  ("child", "a young child playing"), ("crowd", "a crowd of people in the street"), ("adult", "a man standing")])
ZE = embed_texts(model, list(ZS.values()))
mask = EV["test"]["cats"] != "other"; names = list(ZS)
y_true = np.array([names.index(c) for c in EV["test"]["cats"][mask]]); y_pred = (ie_a[torch.from_numpy(mask)] @ ZE.T).argmax(1).numpy()
acc = (y_true == y_pred).mean(); print(f"zero-shot accuracy on {mask.sum()} test images: {acc:.3f} (chance = {1/len(names):.3f})")
cm = confusion_matrix(y_true, y_pred, labels=range(len(names)))
fig, ax = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw=dict(width_ratios=[1.5, 1]))
sns.heatmap(cm / np.maximum(cm.sum(1, keepdims=True), 1), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=names, yticklabels=names, ax=ax[0], cbar=False)
ax[0].set_xlabel("Predicted (closest prompt)"); ax[0].set_ylabel("Keyword theme"); ax[0].set_title(f"Zero-shot confusion matrix: accuracy {acc:.1%}"); ax[0].tick_params(axis="x", rotation=45)
rec = cm.diagonal() / np.maximum(cm.sum(1), 1); ax[1].barh(names, rec, color="tab:green"); ax[1].axvline(1 / len(names), color="red", ls="--", label="chance"); ax[1].set_title("Per-theme recall"); ax[1].legend()
plt.tight_layout(); save_fig("44_zero_shot"); plt.show()

## 15. Summary + save everything

In [ ]:
summary = OrderedDict([
    ("parameters", arch["total"]), ("train minutes", sum(hist["epoch_time"]) / 60), ("final temperature", hist["temperature"][-1]),
    ("test i2t R@1/5/10", [round(res_after[f"i2t_R@{k}"], 3) for k in (1, 5, 10)]), ("test t2i R@1/5/10", [round(res_after[f"t2i_R@{k}"], 3) for k in (1, 5, 10)]),
    ("test median rank i2t / t2i", [res_after["i2t_medr"], res_after["t2i_medr"]]), ("test rsum (before -> after)", [round(res_before["rsum"], 2), round(res_after["rsum"], 2)]),
    ("modality gap (before -> after)", [round(diag["before"]["gap"], 3), round(diag["after"]["gap"], 3)]),
    ("kNN mixing (before -> after)", [round(diag["before"]["knn_mix"], 3), round(diag["after"]["knn_mix"], 3)]),
    ("zero-shot theme accuracy", round(float(acc), 3)),
])
for k, v in summary.items():
    print(f"{k:34s}{v}")

torch.save({"model": best_state, "vocab": itos, "config": dict(EMBED_DIM=EMBED_DIM, TXT_LAYERS=TXT_LAYERS, TXT_HEADS=TXT_HEADS, MAX_LEN=MAX_LEN, IMG_SIZE=IMG_SIZE)}, "clip_flickr8k.pt")
np.savez_compressed("test_embeddings.npz", img=ie_a.numpy(), txt=te_a.numpy(), t2i=EV["test"]["t2i"].numpy(), cats=EV["test"]["cats"])
json.dump({"history": hist, "summary": {k: (v if not isinstance(v, (np.floating,)) else float(v)) for k, v in summary.items()}}, open("results.json", "w"), default=float, indent=1)
shutil.make_archive("clip_images", "zip", "images")
print("\nSaved: clip_flickr8k.pt, test_embeddings.npz, results.json, clip_images.zip (all plots + interactive HTML)")
try:
    from google.colab import files
    files.download("clip_images.zip")
except Exception:
    pass